# Doc cau truc bang Oracle (schema LAKEHOUSE)

Ket noi toi Oracle XE (service `oracle-db` trong docker-compose, port map ra host la `ORACLE_PORT=1521`), doc cau truc 4 bang `CUSTOMERS`, `SALES`, `STORES`, `PRODUCTS` trong schema `LAKEHOUSE`, roi xuat ra file Excel (moi bang 1 sheet).

## 1. Import thu vien

In [ ]:
import oracledb
import pandas as pd

## 2. Ket noi Oracle

`python-oracledb` mac dinh chay thin mode (khong can cai Oracle Instant Client).
Thong tin ket noi lay tu `.env`: `ORACLE_APP_USER` / `ORACLE_APP_USER_PASSWORD`, service name `XEPDB1`, host `localhost` vi notebook chay ngoai docker network, port lay tu `ORACLE_PORT`.

In [ ]:
ORACLE_HOST = "localhost"
ORACLE_PORT = 1521
ORACLE_SERVICE = "XEPDB1"
ORACLE_USER = "lakehouse"
ORACLE_PASSWORD = "aA@123456"
SCHEMA = "LAKEHOUSE"

dsn = oracledb.makedsn(ORACLE_HOST, ORACLE_PORT, service_name=ORACLE_SERVICE)
conn = oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=dsn)
print("Da ket noi Oracle:", conn.version)

## 3. Danh sach bang can doc cau truc

In [ ]:
TABLES = ["CUSTOMERS", "SALES", "STORES", "PRODUCTS"]

## 4. Doc cau truc tung bang tu data dictionary (`ALL_TAB_COLUMNS`)

In [ ]:
COLUMN_SQL = """
SELECT
    column_name,
    data_type,
    data_length,
    data_precision,
    data_scale,
    nullable,
    column_id
FROM all_tab_columns
WHERE owner = :owner
  AND table_name = :table_name
ORDER BY column_id
"""

table_structures = {}

for table_name in TABLES:
    df = pd.read_sql(
        COLUMN_SQL,
        con=conn,
        params={"owner": SCHEMA, "table_name": table_name},
    )
    table_structures[table_name] = df
    print(f"{table_name}: {len(df)} cot")
    display(df)

## 5. Xuat ra file Excel (moi bang 1 sheet)

In [ ]:
OUTPUT_FILE = "oracle_lakehouse_table_structures.xlsx"

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    for table_name, df in table_structures.items():
        df.to_excel(writer, sheet_name=table_name, index=False)

print("Da xuat file:", OUTPUT_FILE)

## 6. Dong ket noi

In [ ]:
conn.close()